<a href="https://colab.research.google.com/github/bilgeerol/bim448-sentiment/blob/main/notebooks/04_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 0. KURULUM HÜCRESİ — her oturumda ilk bu çalıştırılır
from google.colab import userdata, drive
import os
drive.mount('/content/drive')  # modeller ve ham veri Drive'da saklanır
TOKEN = userdata.get('GITHUB_TOKEN')
REPO_OWNER = "bilgeerol"
REPO = "bim448-sentiment"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone https://{TOKEN}@github.com/{REPO_OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git config user.name "sudedereli"
!git config user.email "sudedereli04@gmail.com"

Mounted at /content/drive
Cloning into '/content/bim448-sentiment'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 13 (delta 1), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 12.03 KiB | 2.00 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/bim448-sentiment
Already up to date.


In [2]:
# Depodaki bütün klasör ve dosyaları listele (.git klasörü hariç), son kayıtları göster
!find . -path ./.git -prune -o -print | sort
!git log --oneline -5

.
./data
./data/processed
./data/processed/.gitkeep
./.gitignore
./notebooks
./notebooks/03_bert_finetune.ipynb
./notebooks/.gitkeep
./predictions
./predictions/.gitkeep
./README.md
./report
./report/.gitkeep
./requirements.txt
./results
./results/figures
./results/figures/.gitkeep
./results/hata_ornekleri
./results/hata_ornekleri/.gitkeep
./results/logs
./results/logs/.gitkeep
./results/tables
./results/tables/.gitkeep
./src
./src/.gitkeep
8bbbb90 (HEAD -> main, origin/main, origin/HEAD) Colab kullanılarak oluşturuldu
1cfafee [Bilge] Depo iskeleti, klasörler, requirements ve gitignore eklendi


In [1]:
# Secrets'taki Groq anahtarını oku; güvenlik için anahtarın tamamını değil, sadece başını ve uzunluğunu yazdır
from google.colab import userdata

anahtar = userdata.get("GROQ_API_KEY")
print("Anahtar bulundu.")
print("İlk 4 karakter:", anahtar[:4])
print("Uzunluk:", len(anahtar))

Anahtar bulundu.
İlk 4 karakter: gsk_
Uzunluk: 56


In [2]:
# Groq kütüphanesini kur, gpt-oss-120b'ye tek bir deneme isteği gönder, cevabı ve limit bilgilerini incele
!pip install -q groq

from google.colab import userdata
from groq import Groq

# 1) Groq'a bağlanan "istemci" nesnesini oluştur (anahtar Secrets'tan geliyor)
client = Groq(api_key=userdata.get("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"

# 2) Model bu hesabın model listesinde var mı?
model_listesi = [m.id for m in client.models.list().data]
print("Model listede mi?", MODEL in model_listesi)

# 3) Çok basit bir deneme isteği
#    with_raw_response: cevabın yanında Groq'un gönderdiği HTTP başlıklarını da (limit bilgileri) almamızı sağlar
raw = client.chat.completions.with_raw_response.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with exactly one word: OK"}],
    temperature=0,
)
resp = raw.parse()   # başlıkları ayırdıktan sonra normal cevap nesnesini al

# 4) Cevabı incele
mesaj = resp.choices[0].message
print("\n--- Cevap metni ---")
print(repr(mesaj.content))
print("\n--- Düşünme (reasoning) metni var mı? ---")
print(repr(getattr(mesaj, "reasoning", None)))

# 5) Token kullanımı
print("\n--- Token kullanımı ---")
print(resp.usage)

# 6) Limit başlıkları
print("\n--- Limit başlıkları ---")
for k, v in raw.headers.items():
    if "ratelimit" in k.lower():
        print(f"{k}: {v}")

Model listede mi? True

--- Cevap metni ---
'OK'

--- Düşünme (reasoning) metni var mı? ---
'The user says: "Reply with exactly one word: OK". So they want a single word response: "OK". Must be exactly one word. So answer "OK". Ensure no extra punctuation? "OK" is a word. Should we include newline? Probably just "OK".'

--- Token kullanımı ---
CompletionUsage(completion_tokens=68, prompt_tokens=78, total_tokens=146, completion_time=0.141475872, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=58), prompt_time=0.016491856, prompt_tokens_details=None, queue_time=0.081322255, total_time=0.157967728)

--- Limit başlıkları ---
x-ratelimit-limit-requests: 1000
x-ratelimit-limit-tokens: 8000
x-ratelimit-remaining-requests: 999
x-ratelimit-remaining-tokens: 7662
x-ratelimit-reset-requests: 1m26.4s
x-ratelimit-reset-tokens: 2.535s


In [3]:
# S1: Proje prompt'u ile 10 deneme cümlesini tek istekte gönder, JSON'u kontrol et, iki düşünme seviyesini karşılaştır
import json, time

# --- Rehberdeki SYSTEM prompt'u (kelimesi kelimesine; değiştirme) ---
SYSTEM_PROMPT = """You are a sentiment classifier for product reviews. Each review is written
in English or Turkish. For each review, decide the reviewer's overall
sentiment toward the product or purchase:
- positive: the reviewer is mainly satisfied
- negative: the reviewer is mainly dissatisfied
- neutral: the review is mixed, average/mediocre, or expresses no clear
satisfaction or dissatisfaction
Return only a JSON object of the form
{"results": [{"id": "<id>", "label": "positive|negative|neutral"}]}
containing every given id exactly once, in the same order.
Do not add explanations."""

# --- 10 deneme cümlesi: (id, metin, bizim beklediğimiz etiket) ---
deneme = [
    ("deneme_en_1", "Absolutely love it, works perfectly and arrived early.", "positive"),
    ("deneme_en_2", "Oh great, it broke after two days. Exactly what I wanted.", "negative"),   # ironi
    ("deneme_en_3", "The sound quality is great but the battery dies in an hour.", "neutral"),  # karışık
    ("deneme_en_4", "It's okay. Does the job, nothing special.", "neutral"),
    ("deneme_en_5", "Waste of money, the zipper fell off the first time I used it.", "negative"),
    ("deneme_tr_1", "Harika bir ürün, herkese tavsiye ederim.", "positive"),
    ("deneme_tr_2", "Tam üç günde bozuldu, ne kaliteli ürün ama!", "negative"),                 # ironi
    ("deneme_tr_3", "Kumaşı güzel ama dikişleri biraz özensiz.", "neutral"),                    # karışık
    ("deneme_tr_4", "Fiyatına göre idare eder, ne iyi ne kötü.", "neutral"),
    ("deneme_tr_5", "Kargo çok geç geldi ve ürün kırık çıktı.", "negative"),
]

# --- USER mesajı: rehberdeki biçim "[id] metin" ---
USER_MSG = "Reviews:\n" + "\n".join(f"[{i}] {t}" for i, t, _ in deneme)
print(USER_MSG)

GECERLI = {"negative", "neutral", "positive"}
beklenen_idler = [i for i, _, _ in deneme]

for effort in ["low", "medium"]:
    print(f"\n========== reasoning_effort = {effort} ==========")
    bas = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_MSG},
        ],
        temperature=0,
        response_format={"type": "json_object"},
        extra_body={"reasoning_effort": effort},
    )
    sure = time.time() - bas
    metin = resp.choices[0].message.content
    print("Ham cevap:", metin)

    # 1) JSON açılıyor mu?
    try:
        sonuc = json.loads(metin)["results"]
    except Exception as e:
        print("❌ JSON açılamadı:", e)
        continue

    # 2) Tüm id'ler aynı sırayla var mı?  3) Etiketler geçerli mi?
    gelen_idler = [r.get("id") for r in sonuc]
    print("Tüm id'ler aynı sırayla var mı?", gelen_idler == beklenen_idler)
    print("Geçersiz etiket sayısı:", sum(r.get("label") not in GECERLI for r in sonuc))

    # Tek tek karşılaştırma
    tahmin = {r.get("id"): r.get("label") for r in sonuc}
    for i, t, b in deneme:
        p = tahmin.get(i)
        print(f"{'✓' if p == b else '✗'} {i:12} beklenen={b:8} tahmin={str(p):8} | {t}")

    # 4) Token ve süre
    print(f"Token → girdi: {resp.usage.prompt_tokens}, çıktı: {resp.usage.completion_tokens}, "
          f"toplam: {resp.usage.total_tokens} | süre: {sure:.1f} sn")

Reviews:
[deneme_en_1] Absolutely love it, works perfectly and arrived early.
[deneme_en_2] Oh great, it broke after two days. Exactly what I wanted.
[deneme_en_3] The sound quality is great but the battery dies in an hour.
[deneme_en_4] It's okay. Does the job, nothing special.
[deneme_en_5] Waste of money, the zipper fell off the first time I used it.
[deneme_tr_1] Harika bir ürün, herkese tavsiye ederim.
[deneme_tr_2] Tam üç günde bozuldu, ne kaliteli ürün ama!
[deneme_tr_3] Kumaşı güzel ama dikişleri biraz özensiz.
[deneme_tr_4] Fiyatına göre idare eder, ne iyi ne kötü.
[deneme_tr_5] Kargo çok geç geldi ve ürün kırık çıktı.

========== reasoning_effort = low ==========
Ham cevap: {"results":[{"id":"deneme_en_1","label":"positive"},{"id":"deneme_en_2","label":"negative"},{"id":"deneme_en_3","label":"neutral"},{"id":"deneme_en_4","label":"neutral"},{"id":"deneme_en_5","label":"negative"},{"id":"deneme_tr_1","label":"positive"},{"id":"deneme_tr_2","label":"negative"},{"id":"deneme_tr_

In [4]:
# Secrets'taki GitHub token'ını oku; güvenlik için yalnızca başını ve uzunluğunu yazdır
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
print("Token bulundu.")
print("İlk 4 karakter:", token[:4])
print("Uzunluk:", len(token))

Token bulundu.
İlk 4 karakter: ghp_
Uzunluk: 40
